# Impl 4 — self-distilled replay, on Colab (matched to Impl 3)

Colab driver for `p7/POC/impl4_ssd` (PLAN.md / RUNBOOK.md). It runs the same scripts the
cluster runs — nothing is re-implemented here — in the order `run_all.sh` runs them, with
Slurm replaced by a loop and session-death handled by Drive persistence + `--resume auto`.

**This notebook is configured to be point-for-point comparable with Impl 3** (KL-reweighted
SFT). Impl 4 builds data, trains, and saves checkpoints; the eval is then run by *Impl 3's own
driver*, unmodified, so the numbers land on their axes rather than merely resembling them. That
requires the Impl-3 comparability bundle (`impl3_handoff.tar.gz`) — §4b.

What is matched, and where it is enforced:

| | value | enforced in |
|---|---|---|
| dataset | `meric533/socrateach-sft` @ `1fd0b54a…` — SIs baked in, **never regenerated** | `build_pedagogy_pool.py` |
| steps | 923 (923 blocks × 32), same step numbering as theirs | `impl4/config.py` |
| checkpoints | union of their log grid and PLAN §7's dense grid — 22 points | `impl4/config.py` |
| KL | forward KL(π₀‖π) over 64 contexts truncated before the first tutor turn, ±canonical SI | their `common/kl.py` |
| math | 250 GSM8K items, bare **and** boxed-hint, no pedagogy SI | their `math_logic_prompts.jsonl` |
| new-task | pedagogy NLL over `val[:128]` | their `sweep_ckpt_eval.py` |
| versions | `transformers 5.14.1`, `peft 0.20.0`, `datasets 5.0.1`, `accelerate 1.14.0` | §1 |

**A1 is the gate.** Impl 4's A1 arm is vanilla Impl 2 on the same data as their `impl2-rerun`
baseline, so it should reproduce their published numbers. Train and score A1 *first* — one
comparison validates the canonical SI, the item set, the KL rule, the masking, both compat
shims, the dataset revision and the training config at once (§12).

### Pipeline (per arm)

| # | Stage | Script | Notes |
|---|---|---|---|
| 1 | pedagogy pool (22,500, shared) | `build_pedagogy_pool.py` | downloads SocraTeach |
| 2 | SuperNI prompt pool (12,000, shared) | `build_prompt_pool.py` | slowest data stage |
| 3 | replay slot | `build_general_slot.py` | generation lives here |
| 4 | 24/8 block ordering | `mix_and_order.py` | writes the train file |
| 5 | acceptance checks + loss-norm probe | `acceptance_checks.py`, `probe_loss_norm.py` | PLAN §11 |
| 6 | train + checkpoint grid | `train_sft_impl4.py` | 923 steps, 22 adapters |
| 7 | **matched eval** | Impl 3's `sweep_ckpt_eval.py` | KL / GSM8K / ped-NLL per checkpoint |
| 8 | gate + merge + figure | `impl3_compat/compare.py` | A1 vs their SFT baseline |

### What differs from the cluster run, and why

* **No Slurm.** `run_all.sh` submits one sbatch per arm; here arms run sequentially in a loop.
* **`--runs_root` points at Drive** so adapters survive a disconnect. `--resume auto` picks the
  training back up; `--save_steps` is lowered from 300 to 100 to bound what a disconnect costs.
* **fp16 generation on a T4.** `impl4/generate.py` picks bf16-or-fp32; T4 has no bf16, and fp32
  generation is ~2× slower for no benefit here. §5 below patches that to fp16 *only* when the
  GPU lacks bf16, and prints what it changed. Training already does this on its own
  (`train_sft.py:127` sets `fp16=True` when bf16 is unavailable).
* **`--backend hf`** by default. vLLM is ~10× faster but installing it on Colab usually replaces
  torch and forces a runtime restart — §5b makes that opt-in.
* **`--poc` first.** 63 blocks (~2,000 examples) rehearses the whole pipeline in minutes, which
  is PLAN §11 check 7 and is also the only sane way to find out whether your GPU tier can hold
  a full arm before spending hours on one.

### Rough wall-clock, one arm, full (923 steps)

| GPU | generation (A3, ~8.6k prompts, hf) | training | total |
|---|---|---|---|
| L40S (the cluster) | ~20-30 min | ~40 min | ~1 h |
| A100 40GB | ~25-40 min | ~40-60 min | ~1.5 h |
| L4 | ~50-80 min | ~1.5-2 h | ~3 h |
| T4 (free tier) | ~1-2 h | ~2.5-3.5 h | ~4-5 h |

A1 and A2 need no generation, so they are training-time only. `--poc` is roughly 1/15th of the
training time and ~1/15th of the generation. These are estimates from the model size and step
count, not measurements — check the first arm's `train.log` before planning the matrix.

> **T4 + full run:** one arm exceeds a comfortable free-session window. Either use `--poc`, or
> run one arm per session and rely on `--resume auto`.

### Run order

Three passes, in this order. The config defaults are set for pass 1.

| pass | config | what it proves | cost |
|---|---|---|---|
| **1. smoke** | `POC = True`, `ARMS = ["A1","A3"]` | the pipeline runs end to end, including generation and the checkpoint callback | minutes |
| **2. the A1 gate** | `POC = False`, `ARMS = ["A1"]`, `EVAL_ARMS = ["A1"]` | your setup reproduces Impl 3's published baseline — the one measurement that validates everything at once | one 923-step run + 22 checkpoint evals |
| **3. the matrix** | `POC = False`, `ARMS = ["A1","A2","A3","T4"]` (or all eight) | the actual experiment | see the tables above |

Two things about pass 2 worth knowing:

* **A POC run cannot be compared to Impl 3.** 63 steps is not 923, so its checkpoints share no
  step with theirs. Pass 1 is a rehearsal, not a result.
* **A1 needs no SuperNI pool.** Its replay slot is Tulu gold, so `need_pool` is false and §7 —
  the slowest data stage, 15-30 min of streaming — can be skipped entirely for the gate. Only A2
  and the SSD arms need it.

## 0 · Environment probe

Run this first. `bf16` decides the generation-dtype patch in §5; `Runtime → Change runtime type`
is where you pick the GPU.

In [ ]:
import shutil, subprocess, sys

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE VISIBLE — set Runtime > Change runtime type > GPU")

try:
    import torch
    BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()} | bf16={BF16}")
except ImportError:
    BF16 = False
    print("torch not importable yet — run the install cell, then re-run this one")

print(f"disk free: {shutil.disk_usage('/content' if sys.platform == 'linux' else '.').free / 2**30:.0f} GB")
try:
    with open("/proc/meminfo") as f:
        total = int(next(l for l in f if l.startswith("MemTotal")).split()[1])
    print(f"RAM: {total / 2**20:.0f} GB")
except Exception:
    pass

## 1 · Dependencies — pinned to Impl 3's

**Exact pins, not floors.** Impl 3's handoff is explicit that a version bump mid-sweep makes
runs non-comparable, and the `transformers>=4.48` gradient-accumulation loss fix (PLAN §5) is
load-bearing on our side too. These are their `requirements.txt` values.

Two notes:

* **Never install `torchao`.** Their pitfall: an old version (0.10) breaks
  `peft.get_peft_model`. It is not needed here — this cell does not install it, and nothing
  should.
* **torch is left as Colab's.** Theirs is 2.5.1+cu121; forcing that on Colab risks the whole
  CUDA stack for a difference their own data says is tolerable — a POC-lineage adapter matched
  their baseline to within 1% of axis range across different seeds *and* fp16-vs-bf16. The
  actual version is recorded in every manifest, so the delta is on the record rather than
  hidden.

`langdetect` is Impl 4's own need: `prepare_socrateach_sft.py` uses it for the Tülu English
filter, and without it that filter silently degrades to a script-ratio heuristic.

In [ ]:
!pip -q install "transformers==5.14.1" "datasets==5.0.1" "accelerate==1.14.0" \
                "peft==0.20.0" "huggingface_hub==1.25.1" "numpy==2.4.6" \
                "langdetect==1.0.9" "pyarrow==25.0.0" "matplotlib==3.11.1"

import importlib
PINS = {"transformers": "5.14.1", "datasets": "5.0.1", "accelerate": "1.14.0",
        "peft": "0.20.0", "huggingface_hub": "1.25.1", "numpy": "2.4.6"}
for m, want in PINS.items():
    try:
        got = getattr(importlib.import_module(m), "__version__", "?")
        print(f"  {m:18s} {got:12s} {'OK' if got == want else 'WANTED ' + want}")
    except Exception as e:
        print(f"  {m:18s} MISSING ({e})")
import torch
print(f"  {'torch':18s} {torch.__version__:12s} (Colab's; theirs is 2.5.1+cu121 — recorded, not matched)")
try:
    import torchao
    print("\n  WARNING: torchao is installed. Impl 3 reports 0.10 breaking peft.get_peft_model.\n"
          "           If training fails inside get_peft_model, uninstall it.")
except ImportError:
    pass

## 2 · Google Drive (persistence)

Colab disks are wiped on disconnect, and the deliverable is checkpoints. What goes where:

* **`runs/<arm>/` → Drive.** Adapters, train file, manifest, log. Also what `--resume auto`
  reads, so a session death costs at most `SAVE_STEPS` steps.
* **Derived data → Drive** (~100 MB): the pedagogy pool, the SuperNI prompt pool, the token
  reference files. Restored on the next session instead of rebuilt.
* **The 136 MB SuperNI scan cache and the HF model cache stay local.** Many small files over the
  Drive FUSE mount is slow, and both are cheap to recreate relative to what they'd cost to sync.

Set `USE_DRIVE = False` for a throwaway run.

In [ ]:
import os, shutil
from pathlib import Path

USE_DRIVE = True          # False = everything local, lost on disconnect
DRIVE_SUBDIR = "impl4_ssd"

DRIVE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
    for sub in ("data", "runs", "shared"):
        (DRIVE / sub).mkdir(parents=True, exist_ok=True)
    print("Drive root:", DRIVE)
else:
    print("Drive disabled — checkpoints will not survive a disconnect.")

## 3 · The code

`impl4_ssd` imports its siblings by path — `ORCD-SFT/train_sft.py` for the tokenizer/masking/LoRA
setup, `socrateach_sft/prepare_socrateach_sft.py` for the pedagogy pool and the Tülu loader,
`math_eval/` + `general_eval/` prompt files for the 13-gram decontamination check. So the clone
has to be the whole `p7/POC` tree, not just this directory. That is deliberate (PLAN §8.4: reuse
the Impl 2 code objects rather than copies, so Impl 2 stays reproducible) and it is why this
notebook drives the scripts instead of inlining them.

`tests/test_impl4.py` is stdlib-only and takes about a second — a free check that the clone is
intact before anything downloads a model.

In [ ]:
REPO_URL = "https://github.com/edu-llm/p7stuff.git"
BRANCH = "impl4-ssd"
CLONE = Path("/content/p7stuff")

if not (CLONE / ".git").exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE}
else:
    print("clone present:", CLONE)

POC_ROOT = CLONE / "p7" / "POC"
IMPL4 = POC_ROOT / "impl4_ssd"
PY = "python"

need = [
    IMPL4 / "build_general_slot.py",
    POC_ROOT / "ORCD-SFT" / "train_sft.py",
    POC_ROOT / "ORCD-SFT" / "data" / "socrateach_sft_val.jsonl",
    POC_ROOT / "socrateach_sft" / "prepare_socrateach_sft.py",
    POC_ROOT / "math_eval" / "math_logic_prompts.jsonl",
    POC_ROOT / "general_eval" / "general_prompts.jsonl",
]
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise SystemExit("clone is incomplete, these are required:\n  " + "\n  ".join(missing))

os.chdir(IMPL4)
print("cwd:", os.getcwd())
!git -C {CLONE} log --oneline -1
!{PY} tests/test_impl4.py

## 4 · Configuration

Four things here are not free choices:

* **`ARMS` — A1 first, always.** Every other arm's replay slot is token-matched to A1's Tülu
  slot (`data/tulu_reference.json`), so A1's slot is built as a prerequisite even if you are not
  training A1. Under `TOKEN_REFERENCE = "superni_gold"` the same applies to A2.
* **`PER_DEVICE_BATCH × GRAD_ACCUM == 32`, and `24 % PER_DEVICE_BATCH == 0`.** The 24-pedagogy /
  8-general block layout (PLAN §6) only lands on micro-batch boundaries under those two
  conditions. `8×4` is the cluster's; `4×8` and `2×16` are the safe OOM fallbacks. `16×2` is
  **not** — it would put pedagogy and general in the same micro-batch. Impl 3 runs `32×1` with
  gradient checkpointing off, which needs an H200's memory; the effective batch and the step
  count are identical either way.
* **`MIN_GOLD_WORDS = 25` must be identical across every arm you compare.** It defines the shared
  prompt pool; changing it mid-matrix means A2 and A3 stop drawing from an identical pool and
  stop being a paired control. 25 rather than the plan's 30 is the RUNBOOK §2c recommendation
  (22 tasks / 12 categories, 1.40× token headroom, vs 15 tasks / 9 categories at 30).
* **`TOKEN_REFERENCE` must be identical across compared arms** for the same reason. Both are
  recorded per arm in `manifest.json`; mixing them silently changes what "token-matched" means.

`POC = True` gives 63 blocks instead of 923. POC and full runs are kept in separate run roots
(`runs_poc/` vs `runs/`) and their token-reference files are stashed per mode, so you can flip
back and forth without `--force`-rebuilding anything. A POC run is a pipeline rehearsal only —
its steps do not line up with Impl 3's grid, so it cannot be compared to them.

Not configurable here, because they are matched values living in `impl4/config.py`: **923 steps**
(so step numbers mean the same thing on both sides) and the **22-point union checkpoint grid**
(their 12 log-spaced points plus PLAN §7's dense-early points, so no comparison needs
interpolation). The cell below prints both and asserts the coverage.

In [ ]:
import json, shlex, subprocess, sys, time

# --- what to run -----------------------------------------------------------
POC = True                       # 63 blocks (~2,000 examples). Set False for the real 923.
ARMS = ["A1", "A3"]              # RUNBOOK §7's smoke pair. Full matrix: A1 A2 A3 A4 T2 T3 T4 B2
                                 #   four-run cut: A1 A2 A3 T4   |   one-run cut: A3

# --- shared pool (identical across every arm you compare) ------------------
MIN_GOLD_WORDS = 25              # RUNBOOK §2c
INSTANCES_PER_TASK = 900         # matches the recorded data/superni_pool_meta.json
N_PROMPTS = 12000
TOKEN_REFERENCE = "a1"           # "a1" | "superni_gold"

# --- generation ------------------------------------------------------------
BACKEND = "hf"                   # "hf" | "vllm" (see 5b) | "auto"
GEN_BATCH = 32 if BF16 else 16   # hf backend only; drop to 8 if generation OOMs
CALIBRATE_N = 128 if POC else 256

# --- training (geometry is constrained — see above) ------------------------
PER_DEVICE_BATCH, GRAD_ACCUM = 8, 4
SAVE_STEPS = 100                 # HF resume-only checkpoints; the eval grid is the callback's
SAVE_TOTAL_LIMIT = 1

assert PER_DEVICE_BATCH * GRAD_ACCUM == 32, "block size must stay 32 (PLAN §6)"
assert 24 % PER_DEVICE_BATCH == 0, "24 pedagogy examples must fill whole micro-batches"

MODE = "poc" if POC else "full"
POC_FLAG = "--poc" if POC else ""
RUNS_ROOT = (DRIVE / "runs" / MODE) if DRIVE else (IMPL4 / ("runs_poc" if POC else "runs"))
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RR = shlex.quote(str(RUNS_ROOT))             # the Drive path may contain spaces
COPY_EVAL = "--copy_eval" if DRIVE else ""   # symlinks do not work on the Drive mount

ENV = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")


def sh(cmd, log_path=None, check=True):
    '''Run a command in IMPL4, streaming its output into the notebook.'''
    print("$", cmd, flush=True)
    t0 = time.time()
    fh = open(log_path, "a", encoding="utf-8") if log_path else None
    proc = subprocess.Popen(cmd, shell=True, cwd=IMPL4, env=ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
        if fh:
            fh.write(line)
    rc = proc.wait()
    if fh:
        fh.close()
    dt = time.time() - t0
    print(f"[exit {rc} in {dt/60:.1f} min]", flush=True)
    if check and rc:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")
    return rc


def banner(text):
    print("\n" + "=" * 74 + f"\n== {text}\n" + "=" * 74, flush=True)


def drive_push(*rel):
    if not DRIVE:
        return
    for r in rel:
        src, dst = IMPL4 / r, DRIVE / r
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print(f"  -> Drive: {r}")


def drive_pull(*rel):
    got = []
    if not DRIVE:
        return got
    for r in rel:
        src, dst = DRIVE / r, IMPL4 / r
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            got.append(r)
    if got:
        print("  <- Drive:", ", ".join(got))
    return got


# --- token-reference stashing, so POC and full can coexist ----------------
REF_BASES = ("tulu_reference", "superni_gold_reference")


def activate_reference(mode=MODE):
    '''Put the mode's reference file in place; park any other mode's copy.'''
    for base in REF_BASES:
        live = IMPL4 / "data" / f"{base}.json"
        stash = IMPL4 / "data" / f"{base}.{mode}.json"
        if live.exists():
            other = "poc" if json.loads(live.read_text()).get("poc") else "full"
            if other != mode:
                shutil.copy2(live, IMPL4 / "data" / f"{base}.{other}.json")
                live.unlink()
        if not live.exists() and stash.exists():
            shutil.copy2(stash, live)


def stash_reference(mode=MODE):
    for base in REF_BASES:
        live = IMPL4 / "data" / f"{base}.json"
        if live.exists():
            shutil.copy2(live, IMPL4 / "data" / f"{base}.{mode}.json")
            drive_push(f"data/{base}.{mode}.json")


from impl4.config import CKPT_GRID, IMPL3_LOG_GRID, N_BLOCKS, N_GEN, N_PED, N_TRAIN

print(f"mode={MODE} | arms={ARMS} | runs_root={RUNS_ROOT}")
print(f"backend={BACKEND} gen_batch={GEN_BATCH} | batch {PER_DEVICE_BATCH}x{GRAD_ACCUM}"
      f" | token_reference={TOKEN_REFERENCE} | min_gold_words={MIN_GOLD_WORDS}")
print(f"\nmatched to Impl 3:")
print(f"  {N_BLOCKS} blocks -> {N_PED} pedagogy + {N_GEN} general = {N_TRAIN} rows "
      f"= {N_TRAIN // 32} steps")
print(f"  checkpoint grid ({len(CKPT_GRID)} pts): {list(CKPT_GRID)}")
print(f"  covers their log grid {list(IMPL3_LOG_GRID)}: "
      f"{set(IMPL3_LOG_GRID) <= set(CKPT_GRID)}")

### 4b · The Impl-3 comparability bundle

`impl3_handoff.tar.gz` carries the four things that cannot be reconstructed from their writeup:
`common/prompts/canonical_si.txt` (the exact `+SI` string — `kl_new_SI` is meaningless if it
differs by a clause), the built 250-item GSM8K set, the 1,724-row validation split **in order**
(which pins *which* 64 KL contexts and 128 NLL dialogues get used), and their 194 result rows to
plot against.

Put the tarball on Drive at `MyDrive/<DRIVE_SUBDIR>/impl3_handoff.tar.gz`, or upload it when
prompted. `setup_compat.py` assembles a workdir where **their** eval driver runs unmodified, and
verifies three hashes before letting anything proceed — each one silently invalidates a different
column, so a mismatch is a hard stop rather than a warning.

Run this before training, not after: finding out that the SI differs is cheap now and expensive
once you have eight arms of numbers on the wrong axis.

In [ ]:
banner("4b — Impl 3 comparability bundle")
BUNDLE_TAR = "impl3_handoff.tar.gz"
BUNDLE = Path("/content/impl3_handoff")

if not (BUNDLE / "eval" / "sweep_ckpt_eval.py").exists():
    tar = None
    if DRIVE and (DRIVE / BUNDLE_TAR).exists():
        tar = DRIVE / BUNDLE_TAR
        print(f"found on Drive: {tar}")
    else:
        try:                                    # Colab upload widget
            from google.colab import files
            print(f"Upload {BUNDLE_TAR} (or put it on Drive and re-run this cell):")
            up = files.upload()
            # files.upload() writes into the CURRENT working directory -- §3 chdir'd into
            # impl4_ssd, so it is NOT /content -- and de-duplicates the filename if one is
            # already sitting there ("foo.tar (1).gz"). So resolve the returned key against
            # cwd, then fall back to /content, rather than assuming either.
            name = next((k for k in up if k.endswith(".gz")), None) or next(iter(up), None)
            for cand in ([Path.cwd() / name, Path("/content") / name] if name else []):
                if cand.exists():
                    tar = cand
                    break
            if tar is None and name:            # last resort: it landed somewhere else
                found = sorted(Path("/content").rglob("impl3_handoff.tar*.gz"))
                tar = found[0] if found else None
            if tar and DRIVE:
                shutil.copy2(tar, DRIVE / BUNDLE_TAR)
                print(f"  -> Drive: {BUNDLE_TAR} (skips the upload next session)")
        except ImportError:
            pass
    if tar is None or not Path(tar).exists():
        raise SystemExit(
            f"no {BUNDLE_TAR} found. Put it at MyDrive/{DRIVE_SUBDIR}/{BUNDLE_TAR} and re-run "
            f"this cell, or check where the upload landed:\n"
            f"    sorted(Path('/content').rglob('impl3_handoff.tar*.gz'))")
    print(f"extracting {tar}")
    sh(f"tar xzf {shlex.quote(str(tar))} -C /content")
    if not (BUNDLE / "eval" / "sweep_ckpt_eval.py").exists():
        raise SystemExit(f"extraction did not produce {BUNDLE}/eval/sweep_ckpt_eval.py")
else:
    print(f"bundle present: {BUNDLE}")

os.environ["IMPL3_BUNDLE"] = str(BUNDLE)
sh(f"{PY} impl3_compat/setup_compat.py --bundle {shlex.quote(str(BUNDLE))}")

COMPAT_WORK = IMPL4 / "impl3_compat" / "work"
IMPL3_RESULTS = BUNDLE / "out" / "ckpt_sweep_bare_hint250.jsonl"
print(f"\ncompat workdir : {COMPAT_WORK}")
print(f"their results  : {IMPL3_RESULTS.name} "
      f"({sum(1 for _ in open(IMPL3_RESULTS)) if IMPL3_RESULTS.exists() else 0} rows)")

## 5 · Colab patch — generation dtype

`impl4/generate.py` picks `bfloat16` on capable hardware and `float32` otherwise. On a T4 the
`float32` branch is what runs: ~2× the memory traffic and ~2× the wall-clock for the same
samples. This patches that one fallback to `float16`, and the vLLM `dtype="bfloat16"` literal to
`dtype="auto"`, **only when the GPU lacks bf16**. On an A100/L4 nothing is touched.

Sampling dtype is not free of consequences — fp16 has a narrower range than fp32, so the
sampled text is not bit-identical to what an fp32 run would produce. It is the same tradeoff the
training side already makes (`train_sft.py` runs fp16 on non-bf16 GPUs), and the arms stay
comparable to each other because they all sample under the same dtype. What it does mean: a T4
arm and an A100 arm are **not** bit-comparable, so don't mix GPU tiers within one comparison, and
record which you used. Nothing else in the pipeline is modified.

Re-running this cell is safe.

In [ ]:
gen_py = IMPL4 / "impl4" / "generate.py"
src = orig = gen_py.read_text(encoding="utf-8")

if BF16:
    print("bf16 available — no patch needed.")
else:
    OLD_DTYPE = "dtype = torch.bfloat16 if bf16 else torch.float32"
    NEW_DTYPE = "dtype = torch.bfloat16 if bf16 else torch.float16  # colab: no bf16 on this GPU"
    n = src.count(OLD_DTYPE)
    if n:
        src = src.replace(OLD_DTYPE, NEW_DTYPE)
        print(f"patched hf generation dtype fp32 -> fp16 ({n} site(s))")
    OLD_VLLM = 'dtype="bfloat16"'
    NEW_VLLM = 'dtype="auto"  # colab: no bf16 on this GPU'
    n = src.count(OLD_VLLM)
    if n:
        src = src.replace(OLD_VLLM, NEW_VLLM)
        print(f"patched vLLM dtype bfloat16 -> auto ({n} site(s))")
    if src != orig:
        gen_py.write_text(src, encoding="utf-8")
    elif NEW_DTYPE in orig:
        print("already patched.")
    else:
        print("WARNING: patch targets not found — impl4/generate.py may have changed upstream. "
              "Check it before trusting a T4 generation run.")

!git -C {CLONE} --no-pager diff --stat -- p7/POC/impl4_ssd/impl4/generate.py

### 5b · Optional: vLLM

~10× faster generation (a 1B model over 8.6k prompts is minutes, not hours), and the pipeline
already drives it through `prompt_token_ids` so the §4 formatting invariant holds either way.

The catch is the install: vLLM pins its own torch build and on Colab that usually means torch is
replaced and **the runtime has to restart**, after which you re-run §0-§5 (the clone and Drive
data survive; the pip installs in §1 do not). Worth it for a full arm, not for a `--poc`.

Skipped unless you set `INSTALL_VLLM = True`.

In [ ]:
INSTALL_VLLM = False

if INSTALL_VLLM:
    !pip -q install "vllm>=0.6.0"
    print("\nIf torch was replaced above: Runtime > Restart session, then re-run 0-5, "
          "and set BACKEND = 'vllm' in 4.")
else:
    print("skipped (INSTALL_VLLM = False) — using BACKEND =", BACKEND)

## 6 · Stage 1 — pedagogy pool (shared by every arm)

22,500 pedagogy rows pulled from **`meric533/socrateach-sft` at revision `1fd0b54a…`**, plus the
val/test splits copied across verbatim.

**The system instructions are not regenerated, and that is the single most important thing in
this notebook.** The per-dialogue SIs are baked into the published rows. Impl 3 trains on them
as-is; `prepare_socrateach_sft.py` would generate *different* strings, and pedagogy NLL would
quietly stop measuring the same quantity — with nothing downstream able to detect it. Hence the
pinned revision, and hence `--regenerate` existing only as a loudly-warned escape hatch.

Order matters too: their KL and NLL probes are the **first 64 / first 128 rows of the validation
split in file order**, so val is copied byte-for-byte rather than re-serialised through anything
that might reorder it. (Verified: the Hub val file, the bundle's copy, and `ORCD-SFT/data/`'s
copy are all the same sha256.)

In [ ]:
banner("stage 1 — pedagogy pool")
PED_FILES = [f"data/pedagogy_pool/socrateach_sft_{s}.jsonl" for s in ("train", "val", "test")]
PED_META = "data/pedagogy_pool/pool_source.json"

drive_pull(*PED_FILES, PED_META)
if not (IMPL4 / PED_FILES[0]).exists():
    sh(f"{PY} build_pedagogy_pool.py")
    drive_push(*PED_FILES, PED_META)
else:
    n = sum(1 for _ in open(IMPL4 / PED_FILES[0], encoding="utf-8"))
    print(f"pedagogy pool present: {n} examples")

# The pool must have come from the Hub, not from a local regeneration, or every new-task
# number is on a different footing from Impl 3's.
src = json.loads((IMPL4 / PED_META).read_text()) if (IMPL4 / PED_META).exists() else {}
print(f"  source: {src.get('mode')} {src.get('dataset') or ''} {(src.get('revision') or '')[:12]}")
if not src.get("comparable_to_impl3", False):
    raise SystemExit(
        "This pedagogy pool has locally regenerated system instructions, so pedagogy NLL is NOT "
        "comparable to Impl 3. Rebuild it:\n"
        "    python build_pedagogy_pool.py --force")

## 7 · Stage 2 — SuperNI prompt pool (shared by every arm)

12,000 prompts drawn round-robin from the SuperNI English train tasks that survive: the
contamination blocklist (BIG-Bench / GSM8K / MATH / AIME — BBH ⊂ BIG-Bench and the eval team
grades BBH), a per-instance 13-gram check against `math_logic_prompts.jsonl` and
`general_prompts.jsonl`, and the ≥`MIN_GOLD_WORDS`-word gold-length filter. Building it once is
what makes A2 (gold) and A3 (self-generated) a paired control rather than two samples.

This is the slowest data stage. Three routes, tried in this order:

1. **Drive** — a pool built in an earlier session, or one you copied up from the cluster/laptop
   (`data/superni_pool.jsonl`, ~12 MB, plus `data/superni_pool_meta.json`). Instant, and it
   reproduces the exact pool those runs used. Meta is checked against `MIN_GOLD_WORDS` and you
   get a loud warning on mismatch.
2. **`SUPERNI_ROUTE = "clone"`** — `git clone --depth 1` of `allenai/natural-instructions`, then
   scan locally. The repo is ~4.9 GB with history; a depth-1 clone is a good deal less but still
   the heaviest download here. One fast transfer instead of 757 HTTP requests.
3. **`"stream"`** (the default fallback) — read each task file from the pinned commit over HTTP
   and stop after `INSTANCES_PER_TASK` instances, which turns a ~2 GB pull into ~200 MB across
   757 requests. Budget 15-30 min. The scan caches to `data/superni_cache/` (~136 MB, local
   only), so re-scanning at a different threshold afterwards is free.

`INSTANCES_PER_TASK` changes *which* instances exist to sample (the reader takes each file's
first K), so it changes the pool. 900 matches the recorded `superni_pool_meta.json`; leave it
alone if you want the same pool as the cluster.

In [ ]:
banner("stage 2 — SuperNI prompt pool")
SUPERNI_ROUTE = "auto"          # "auto" | "clone" | "stream"
POOL_FILES = ["data/superni_pool.jsonl", "data/superni_pool_meta.json"]
POOL = IMPL4 / POOL_FILES[0]

if SUPERNI_ROUTE == "auto":
    drive_pull(*POOL_FILES)

if POOL.exists():
    meta = json.loads((IMPL4 / POOL_FILES[1]).read_text()) if (IMPL4 / POOL_FILES[1]).exists() else {}
    n = sum(1 for _ in open(POOL, encoding="utf-8"))
    print(f"pool present: {n} prompts | min_gold_words={meta.get('min_gold_words')} "
          f"| instances_per_task={(meta.get('source') or {}).get('instances_per_task')} "
          f"| tasks_retained={(meta.get('stats') or {}).get('tasks_retained')}")
    if meta.get("min_gold_words") not in (None, MIN_GOLD_WORDS):
        print(f"  WARNING: pool was built at min_gold_words={meta['min_gold_words']} but this "
              f"session is configured for {MIN_GOLD_WORDS}. Arms built against different pools "
              f"are not a paired control. Delete the pool and rebuild, or change MIN_GOLD_WORDS.")
else:
    src_flag = ""
    if SUPERNI_ROUTE == "clone":
        NI = Path("/content/natural-instructions")
        if not (NI / ".git").exists():
            !git clone --depth 1 https://github.com/allenai/natural-instructions {NI}
        src_flag = f"--superni_dir {shlex.quote(str(NI))}"
    sh(f"{PY} build_prompt_pool.py --min_gold_words {MIN_GOLD_WORDS} "
       f"--instances_per_task {INSTANCES_PER_TASK} --n_prompts {N_PROMPTS} {src_flag}")
    drive_push(*POOL_FILES, "shared/superni_train_task_ids.txt")

## 8 · Stage 3 — the replay slot, per arm

The only stage that differs between arms:

| Arm | Replay slot | Cost here |
|---|---|---|
| `A1` | 7,496 Tülu-3 gold — **defines the token budget every other arm matches** | parquet shard download |
| `A2` | 7,496 SuperNI gold | none |
| `A3`=`T1` | 7,496 SSD, `T=1.0`, no truncation | generation |
| `A4` | 3,748 Tülu gold + 3,748 SSD at `T1` | generation (half) |
| `T2`/`T3`/`T4` | SSD at `T=1.0`/`1.3`/`1.6`, `k=20, p=0.8` | generation |
| `B2` | SSD at `T1`, gated against gold (resample ≤4, never fall back to gold) | generation + resampling |

A1's slot is built first whatever `ARMS` says — without `data/tulu_reference.json` every other
arm refuses to build. For the SSD arms, `max_tokens` auto-calibrates toward the Tülu slot's mean
target length; do not hand it `--max_tokens 400` because PLAN §4 guessed 300-500, the measured
value is ~80 (RUNBOOK §2b).

Watch the printed **token ratio**. Outside ±5% it is a real confound, not rounding: under
token-mean loss the arm's replay pressure per step differs from the reference's.

In [ ]:
banner("stage 3 — replay slots")
activate_reference()
drive_pull(*[f"data/{b}.{MODE}.json" for b in REF_BASES])
activate_reference()

SLOT_ARGS = (f"--runs_root {RR} --token_reference {TOKEN_REFERENCE} {POC_FLAG} "
             f"--backend {BACKEND} --batch_size {GEN_BATCH} --calibrate_n {CALIBRATE_N}")

# The token references must exist before any arm that matches to them (run_all.sh:55-65).
if not (IMPL4 / "data/tulu_reference.json").exists():
    banner("prerequisite — A1's Tulu reference slot (defines the token budget)")
    sh(f"{PY} build_general_slot.py --arm A1 {SLOT_ARGS}")
    stash_reference()
if TOKEN_REFERENCE == "superni_gold" and not (IMPL4 / "data/superni_gold_reference.json").exists():
    banner("prerequisite — A2's SuperNI-gold reference slot")
    sh(f"{PY} build_general_slot.py --arm A2 {SLOT_ARGS}")
    stash_reference()

for arm in ARMS:
    banner(f"{arm} — replay slot")
    sh(f"{PY} build_general_slot.py --arm {arm} {SLOT_ARGS}")
stash_reference()

## 9 · Stage 4 — mix and order

Writes `runs/<arm>/socrateach_sft_train.jsonl` as repeating 32-example blocks of **24 pedagogy
then 8 general**, and puts `socrateach_sft_{val,test}.jsonl` beside it (the trainer needs all
three side by side). With `SequentialSampler` and `per_device_batch=8`, micro-batches are
consecutive slices, so positions 0-23 are three pedagogy micro-batches and 24-31 is the general
one — the replay stream becomes a per-step constraint instead of an in-expectation one.

The printed first-3-blocks dump is PLAN §11 check 5; it should read 24 `P` then 8 `g`.

In [ ]:
banner("stage 4 — mix and order")
for arm in ARMS:
    sh(f"{PY} mix_and_order.py --arm {arm} --runs_root {RR} {POC_FLAG} {COPY_EVAL}")

## 10 · Stage 5 — acceptance checks (PLAN §11)

1. the unmasked label span decodes to exactly assistant content + EOS
2. **the generation prompt is byte-identical to the training prefix** — the §4 invariant; if this
   fails the targets are not on-policy w.r.t. π₀ and the whole premise is void
3. no general record has a system message; every pedagogy record has one
4. the loss-normalisation probe — which normalisation the Trainer *actually* uses, since the
   `transformers>=4.48` fix can silently fall back to per-micro-batch mean under PEFT. The answer
   depends on the installed `transformers`, so it is measured rather than assumed, and lands in
   the manifest as `loss_normalization.verdict`. ~2 min per arm (loads the model + LoRA, one step
   at `lr=0`). PLAN §10 wants it in every arm's manifest, so it runs per arm by default — the
   verdict should be identical across arms, and one that differs means something in the
   environment moved mid-matrix.
5. the ordered train file's 24/8 layout
6. zero 13-gram overlap with the eval prompt sets

Check 7 is the `--poc` smoke run — that is `POC = True` in §4, i.e. this whole notebook.

**A failure here means stop.** These checks exist because each of them, if broken, invalidates the
experiment silently rather than loudly.

In [ ]:
banner("stage 5 — acceptance checks")
PROBE_EVERY_ARM = True    # False = probe once; the other manifests then carry no verdict

for i, arm in enumerate(ARMS):
    probe = "--with_probe" if (PROBE_EVERY_ARM or i == 0) else ""
    sh(f"{PY} acceptance_checks.py --arm {arm} --runs_root {RR} {probe}")

## 11 · Stage 6 — train

Impl 2's trainer, imported not copied (`make_tokenize_fn` / `load_model_and_tokenizer` are the
same code objects), plus exactly four changes: `SequentialSampler`, `dataloader_drop_last=True`,
the dense checkpoint-grid callback, and `--arm`.

Checkpoints land at the 22-point union grid — **1, 2, 3, 4, 5, 8, 10, 16, 20, 32, 40, 64, 80,
128, 160, 256, 320, 480, 512, 640, 800, 923** (POC: 5, 10, 20, 40, 63). Impl 3's 12 log-spaced
points are all in there, so every checkpoint of theirs has a matching one of ours.
The early points are the whole reason for the grid — forgetting is concentrated in the first ~20
steps, where a uniform `save_steps` grid sees nothing. They sit *inside* warmup
(0.03 × 923 ≈ 28 steps), which is intentional and flagged in the manifest: that is where the
damage happens, but they are not points on the cosine schedule and shouldn't be read as such.

**If the session dies, re-run this cell.** `--resume auto` picks up the last HF checkpoint
(≤`SAVE_STEPS` steps lost), and grid adapters already on disk are not rewritten. Adapters are
~25 MB each; the HF resume checkpoints are larger and capped at `SAVE_TOTAL_LIMIT`.

`train.log` is teed into `runs/<arm>/` as the RUNBOOK expects.

In [ ]:
banner("stage 6 — train")
for arm in ARMS:
    out = RUNS_ROOT / arm
    out.mkdir(parents=True, exist_ok=True)
    banner(f"{arm} — training")
    sh(f"{PY} train_sft_impl4.py --arm {arm} --runs_root {RR} {POC_FLAG} "
       f"--resume auto --per_device_batch {PER_DEVICE_BATCH} --grad_accum {GRAD_ACCUM} "
       f"--save_steps {SAVE_STEPS} --save_total_limit {SAVE_TOTAL_LIMIT}",
       log_path=out / "train.log")

## 12 · Matched eval — Impl 3's driver on our checkpoints

Three axes, all measured by **their** `eval/sweep_ckpt_eval.py`, unmodified:

| axis | what |
|---|---|
| `kl_new_SI`, `kl_ped_noSI` | forward KL(π₀‖π) per token over base-greedy continuations of 64 held-out pedagogy contexts, each truncated **before the first tutor turn**, with and without the canonical SI |
| `math_bare`, `math_hint` | 250 GSM8K items, integer exact match, question alone vs question + `Put your final answer inside \boxed{ }.` — neither carrying a pedagogy SI |
| `ped_nll` | mean per-token NLL of the gold tutor turns over `val[:128]` |

Plus `commit` and `deflect` per condition. Their warning applies: on this item set `commit` is
contaminated — the extractor pulls a number out of a Socratic counter-question and marks it wrong
— so **`deflect` is the trustworthy refusal signal** and `acc_given_commit` should not be read
when deflect is high.

**Run A1 alone first.** It is vanilla Impl 2 on the same data as their `impl2-rerun`, so it
should reproduce their published row; one comparison validates the SI, the item set, the KL rule,
the masking, both shims, the dataset revision and the training config together. Only then spend
GPU time on the other arms.

### What this costs

Their cold pass is ~3.5 h for 192 checkpoints on an H200 (~65 s each). Scaled:

| GPU | per checkpoint | A1 only (22) | 4-run cut (88) | all 8 arms (176) |
|---|---|---|---|---|
| A100 | ~3-4 min | ~1.3 h | ~5 h | ~10 h |
| L4 | ~6-8 min | ~2.5 h | ~10 h | ~20 h |
| T4 | ~10-15 min | ~4.5 h | ~18 h | ~35 h |

The driver appends per checkpoint and skips ones already scored, so a disconnect costs one
checkpoint. Use `EVAL_STEPS` to score only their 12 log points when the budget is tight — but
**never** shrink `--gen_max` or the item count: both are inside the protocol hash, and changing
either silently makes the rows unmergeable.

In [ ]:
banner("stage 7 — matched eval (Impl 3's driver)")

EVAL_ARMS = ["A1"]        # start here. Widen once the gate passes.
EVAL_STEPS = None         # None = the full 22-point union grid; or "1,2,3,4,8,16,32,64,128,256,512,923"
EVAL_BATCH = 32 if BF16 else 16

CW = shlex.quote(str(COMPAT_WORK))
RESULTS = COMPAT_WORK / "out" / "ckpt_sweep_impl4.jsonl"

# 1. expose runs/<arm>/ckpt-N as out/impl4-<arm>/checkpoint-N (symlinks; runs/ is not touched).
#    Arms that did not reach their final grid step are skipped and named — Impl 3's epoch>=0.99
#    guard exists because two of their runs were graded while incomplete.
bridge = f"{PY} impl3_compat/bridge.py --runs_root {RR} --workdir {CW}"
if EVAL_ARMS:
    bridge += " --arms " + " ".join(EVAL_ARMS)
if EVAL_STEPS:
    bridge += f" --steps {EVAL_STEPS}"
sh(bridge)

# 2. their driver. Appends as it goes and skips what is already scored, so re-running after a
#    disconnect resumes.
sh(f"cd {CW} && {PY} eval/sweep_ckpt_eval.py --runs 'out/*' "
   f"--out out/ckpt_sweep_impl4.jsonl --batch {EVAL_BATCH}",
   log_path=COMPAT_WORK / "out" / "eval.log")

if DRIVE and RESULTS.exists():
    (DRIVE / "impl3_compat").mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESULTS, DRIVE / "impl3_compat" / RESULTS.name)
    print(f"  -> Drive: impl3_compat/{RESULTS.name}")

### 12b · The A1 gate, and the merged figure

`compare.py` refuses to merge rows whose `protocol` stamps differ, prints A1 against their
`impl2-rerun` row with each delta as a share of the axis range measured from their own 194 rows,
writes a merged JSONL in their schema, and draws a two-panel figure that separates the two
projects.

Reading a miss:

| symptom | first suspect |
|---|---|
| `ped_nll` off, KL and math fine | the `common/chat.py` masking shim, or the dataset revision |
| `kl_*` off | `canonical_si.txt`, or the KL contexts |
| `math_*` off | the item set, or generation settings |
| everything slightly off | ordering (our 24/8 blocks vs their shuffle), or dtype |

Their own `eval/plot_figure3.py` is the canonical figure and will read the merged file, but it
styles every `variant=null` run as a black X — so our arms collide there until they add
`MARKER = {"a": "s", "b": "o", "impl4": "^"}`.

In [ ]:
rc = sh(f"{PY} impl3_compat/compare.py --impl3 {shlex.quote(str(IMPL3_RESULTS))} "
        f"--impl4 {shlex.quote(str(RESULTS))} "
        f"--out {shlex.quote(str(COMPAT_WORK / 'out/merged.jsonl'))} "
        f"--fig {shlex.quote(str(COMPAT_WORK / 'out/impl3_vs_impl4.png'))}",
        check=False)
print("\nGATE PASSED" if rc == 0 else "\nGATE FLAGGED — see the table above before trusting other arms")

from IPython.display import Image, display
fig = COMPAT_WORK / "out" / "impl3_vs_impl4.png"
if fig.exists():
    display(Image(filename=str(fig)))
if DRIVE:
    for f in ("merged.jsonl", "impl3_vs_impl4.png"):
        p = COMPAT_WORK / "out" / f
        if p.exists():
            shutil.copy2(p, DRIVE / "impl3_compat" / f)

## 13 · Deliverables

Per arm, in `runs/<arm>/`: the grid adapters, the exact ordered train file, the replay slot with
full provenance, `manifest.json`, `checkpoint_index.json`, `train.log`. Shared, once:
`shared/superni_train_task_ids.txt` (so the eval team can keep their sets clean).

The numbers to read before handing anything off:

* **`token_ratio`** — outside ±5% the arm's replay pressure differs from the reference's.
* **`loss_norm`** — `token_mean` means token-matching is the binding constraint;
  `micro_batch_mean` means the 24/8 block layout is doing the work instead.
* **`degen`** / **`gate`** drop rates — `T` and `ρ` change output length and junk rate, so a `T`
  comparison across arms with different realized token weights is not a `T` comparison.
* **`ckpts`** — anything missing from the grid needs investigating before handoff; re-running to
  recover a checkpoint costs far more than the disk did.

In [ ]:
rows = []
for arm in ARMS:
    m = json.loads((RUNS_ROOT / arm / "manifest.json").read_text(encoding="utf-8"))
    gs, ssd, tr = m.get("general_slot", {}), (m.get("general_slot", {}) or {}).get("ssd") or {}, m.get("training", {})
    rows.append({
        "arm": m.get("arm"), "block": m.get("block"), "sigma": m.get("sigma"),
        "sampling": (m.get("sampling") or {}).get("name"),
        "slot_tokens": gs.get("total_label_tokens"),
        "token_ratio": gs.get("token_ratio_to_A1"),
        "in_tol": gs.get("within_token_tolerance"),
        "degen": ssd.get("degeneracy_drop_rate"), "gate": ssd.get("gate_drop_rate"),
        "loss_norm": (m.get("loss_normalization") or {}).get("verdict"),
        "steps": tr.get("steps"), "ckpts": len(tr.get("checkpoints_saved") or []),
        "n_fail": (m.get("acceptance") or {}).get("n_fail"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index("arm"))
except ImportError:
    for r in rows:
        print(r)

print()
for arm in ARMS:
    d = RUNS_ROOT / arm
    idx = d / "checkpoint_index.json"
    if idx.exists():
        ci = json.loads(idx.read_text())
        print(f"{arm}: saved {ci['checkpoints_saved']}")
        print(f"     priority (for the eval team) {ci['priority_checkpoints']}")
        missing = [s for s in ci["checkpoint_grid"]
                   if s <= (ci.get("steps") or 0) and s not in ci["checkpoints_saved"]]
        if missing:
            print(f"     WARNING: grid steps {missing} were NOT written")
!du -sh "{RUNS_ROOT}"/*

### Optional — the held-out prompt file

`shared/superni_heldout_prompts.jsonl` is the untouched `test_tasks.txt` split, shipped unused in
case the eval team wants a general-prompt KL axis (PLAN §10). It costs another scan — 119 tasks,
none of them cached — so it is off by default.

In [ ]:
BUILD_HELDOUT = False

if BUILD_HELDOUT:
    sh(f"{PY} build_prompt_pool.py --split test --min_gold_words {MIN_GOLD_WORDS} "
       f"--instances_per_task {INSTANCES_PER_TASK}")
    drive_push("shared/superni_heldout_prompts.jsonl", "shared/superni_heldout_meta.json")
else:
    print("skipped (BUILD_HELDOUT = False)")

In [ ]:
# Archive the manifests + the shared handoff files (small; the adapters are already on Drive).
import datetime, tarfile

if DRIVE:
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
    tarball = DRIVE / f"impl4_{MODE}_manifests_{stamp}.tar.gz"
    with tarfile.open(tarball, "w:gz") as tf:
        for arm in ARMS:
            for name in ("manifest.json", "checkpoint_index.json", "train.log"):
                p = RUNS_ROOT / arm / name
                if p.exists():
                    tf.add(p, arcname=f"{arm}/{name}")
        for p in (IMPL4 / "shared").glob("*"):
            tf.add(p, arcname=f"shared/{p.name}")
        meta = IMPL4 / "data/superni_pool_meta.json"
        if meta.exists():
            tf.add(meta, arcname="data/superni_pool_meta.json")
    print("wrote", tarball)
else:
    print("Drive disabled — nothing archived.")

## 14 · Things that will bite

**Comparability with Impl 3 — the ones that fail silently**

* **A regenerated pedagogy pool.** If `pool_source.json` says `regenerated`, the system
  instructions differ from the published rows and every new-task number is on its own footing.
  §6 hard-stops on this; don't route around it.
* **A different bundle.** `setup_compat.py` verifies the canonical SI, the 250 item ids and the
  val split. If any hash moves, stop — the axis moved with it.
* **Mixed protocol stamps.** `compare.py` refuses rather than merging. If it fires, re-run the
  odd side; do not edit the stamp.
* **Shrinking the eval to save time.** `--gen_max 512` and the 250 items are inside the protocol
  hash. Score fewer *checkpoints* (`EVAL_STEPS`), never a cheaper probe.
* **Reporting one math number.** Bare and hinted differ by ~24 points on an SFT checkpoint and
  by ~0 on base. Always say which — their pitfall #2.
* **Reading `acc_given_commit` when deflect is high.** The extractor scores a number lifted out
  of a Socratic counter-question, so a refusal counts as a committed wrong answer. Use `deflect`.
* **Mixing GPU tiers** if §5 patched the generation dtype on one of them.

**Colab-specific**

* **Session died mid-training** → re-run §11. `--resume auto` handles it; nothing else needs redoing.
* **Session died mid-generation** → re-run §8. There is no partial-generation resume, so the arm's
  slot restarts from scratch. On a T4 that is an hour you don't want to lose twice — consider
  `--poc` or vLLM (§5b) first.
* **CUDA OOM in training** → `PER_DEVICE_BATCH, GRAD_ACCUM = 4, 8` (or `2, 16`). Both keep the
  32-example block and whole-micro-batch pedagogy/general split. Never `16, 2`.
* **CUDA OOM in generation** → lower `GEN_BATCH`. It affects speed only, not the targets.
* **Ran out of Drive quota** → the grid adapters are the deliverable (~25 MB × 11 × arms); the HF
  resume checkpoints are the bulk and are disposable once an arm finishes.
* **Fresh session, want to keep going** → run §0-§7 (fast, restores from Drive), then the stage
  you were on.
* **Matched eval died mid-sweep** → re-run §12. The driver appends per checkpoint and skips what
  is already in the results file, so at most one checkpoint is lost.
* **Garbled progress bars.** Stage output is captured through a pipe, so tqdm's carriage-return
  updates arrive as one long line instead of a bar. Cosmetic. Real progress signals are the
  script's own `log()` lines and the Trainer's loss line every 20 steps.

**Pipeline-level (RUNBOOK §10)**

* **Run A1 first.** Everything is token-matched to it; §8 enforces this.
* **`TOKEN_REFERENCE` and `MIN_GOLD_WORDS` must be identical across every arm you compare.**
  Both are recorded per arm in `manifest.json` — check them there rather than trusting this
  notebook's config cell, which someone may have edited between arms.
* **`--poc` is sticky.** A POC slot has 504 examples, not 7,496, and `mix_and_order.py` refuses to
  mix one into a full run. Flipping `POC` here switches run roots and reference stashes for you.
* **The 5/10/20 checkpoints sit inside warmup.** Intentional — that's where the damage happens —
  but they are not points on the cosine schedule.
* **Anchor staleness is correct.** π₀ is frozen, so by step 900 the replay data is far from θ_t.
  Do not "fix" this by regenerating mid-run, and do not call Impl 4 on-policy self-distillation.
* **Don't mix GPU tiers within a comparison** if §5 patched the dtype on one of them.

**Still out of scope on purpose:** the LLM-as-judge pedagogy rubric. PLAN.md assigns judged
quality to another team, and Impl 3 doesn't have it scored either (their new-task claims all rest
on NLL, with judge batches generated but ungraded — and they report two supposedly-identical SFT
runs landing 0.11 apart on the judge, so treat sub-0.1 judge differences as noise). The matched
eval in §12 is deterministic: KL, GSM8K exact-match, and NLL. No judge anywhere in it.